Linear Classification

In [13]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    GridSearchCV, cross_val_score, train_test_split
)
from sklearn.preprocessing import StandardScaler


In [14]:
data = pd.read_csv(
    "M:/accenture/ds-ml-training/ds-ai-assignments-maksym-bondar/weeks/week1/data/day.csv"
)

Prepare Features

In [21]:
y = data["cnt"]

X = data.drop(["cnt", "instant", "dteday", "registered", "casual"], axis=1)

In [22]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.3, random_state=17
)

In [23]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_holdout_scaled = scaler.transform(X_holdout)

Linear Regression (OLS)

In [24]:
linreg = LinearRegression()
linreg.fit(X_train_scaled, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


MSE

In [25]:
print("MSE (train): {:.3f}".format(
    mean_squared_error(y_train, linreg.predict(X_train_scaled))
))
print("MSE (test): {:.3f}".format(
    mean_squared_error(y_holdout, linreg.predict(X_holdout_scaled))
))

MSE (train): 660547.723
MSE (test): 993748.621


Coefficients Sorted by Importance

In [26]:
linreg_coef = pd.DataFrame(
    {"coef": linreg.coef_, "coef_abs": np.abs(linreg.coef_)},
    index=X.columns
)

linreg_coef.sort_values("coef_abs", ascending=False)


,coef,coef_abs
yr,987.602578,987.602578
season,647.726303,647.726303
atemp,608.925957,608.925957
temp,400.567209,400.567209
weathersit,-298.413577,298.413577
hum,-225.136893,225.136893
mnth,-220.252441,220.252441
windspeed,-217.750940,217.750940
weekday,173.735344,173.735344
holiday,-55.737960,55.737960


Lasso Regression (α = 0.01)

In [27]:
lasso1 = Lasso(alpha=0.01, random_state=17)
lasso1.fit(X_train_scaled, y_train)


,alpha,0.01
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,17
,selection,'cyclic'


In [28]:
lasso1_coef = pd.DataFrame(
    {"coef": lasso1.coef_, "coef_abs": np.abs(lasso1.coef_)},
    index=X.columns
)

lasso1_coef.sort_values("coef_abs", ascending=True).head()

,coef,coef_abs
workingday,50.130622,50.130622
holiday,-55.738271,55.738271
weekday,173.714277,173.714277
windspeed,-217.764750,217.764750
mnth,-220.193371,220.193371


Tuned Lasso (LassoCV)

In [29]:
alphas = np.logspace(-6, 2, 200)
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=17)
lasso_cv.fit(X_train_scaled, y_train)

,eps,0.001
,n_alphas,'deprecated'
,alphas,array([1.0000...00000000e+02])
,fit_intercept,True
,precompute,'auto'
,max_iter,1000
,tol,0.0001
,copy_X,True
,cv,5
,verbose,False
,n_jobs,None


In [30]:
lasso_cv.alpha_

np.float64(2.704959730463137)

In [ ]:
lasso_cv_coef = pd.DataFrame(
    {"coef": lasso_cv.coef_, "coef_abs": np.abs(lasso_cv.coef_)},
    index=X.columns
)

lasso_cv_coef.sort_values("coef_abs", ascending=False)

,coef,coef_abs
yr,984.927642,984.927642
season,632.145805,632.145805
atemp,604.809236,604.809236
temp,403.948477,403.948477
weathersit,-297.679628,297.679628
hum,-222.499653,222.499653
windspeed,-214.866073,214.866073
mnth,-204.273049,204.273049
weekday,170.695243,170.695243
holiday,-54.559712,54.559712


MSE

In [32]:
print("MSE (train): {:.3f}".format(
    mean_squared_error(y_train, lasso_cv.predict(X_train_scaled))
))
print("MSE (test): {:.3f}".format(
    mean_squared_error(y_holdout, lasso_cv.predict(X_holdout_scaled))
))

MSE (train): 660676.025
MSE (test): 989136.478


Random Forest

In [33]:
forest = RandomForestRegressor(random_state=17)
forest.fit(X_train_scaled, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


MSE

In [34]:
print("MSE (train): {:.3f}".format(
    mean_squared_error(y_train, forest.predict(X_train_scaled))
))

print("MSE (cv): {:.3f}".format(
    np.mean(np.abs(cross_val_score(
        forest, X_train_scaled, y_train,
        scoring="neg_mean_squared_error"
    )))
))

print("MSE (test): {:.3f}".format(
    mean_squared_error(y_holdout, forest.predict(X_holdout_scaled))
))

MSE (train): 68004.964
MSE (cv): 524137.888
MSE (test): 440696.268


Hyperparameter Tuning (GridSearchCV)

In [35]:
forest_params = {
    "max_depth": list(range(10, 25)),
    "max_features": list(range(6, 12))
}

locally_best_forest = GridSearchCV(
    RandomForestRegressor(random_state=17, n_jobs=-1),
    forest_params,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=True
)

locally_best_forest.fit(X_train_scaled, y_train)


Fitting 5 folds for each of 90 candidates, totalling 450 fits


,estimator,RandomForestR...ndom_state=17)
,param_grid,"{'max_depth': [10, 11, ...], 'max_features': [6, 7, ...]}"
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,True
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [36]:
locally_best_forest.best_params_
locally_best_forest.best_score_

np.float64(-494264.2982225851)

MSE

In [37]:
print("MSE (cv): {:.3f}".format(
    np.mean(np.abs(cross_val_score(
        locally_best_forest.best_estimator_,
        X_train_scaled, y_train,
        scoring="neg_mean_squared_error"
    )))
))

print("MSE (test): {:.3f}".format(
    mean_squared_error(y_holdout, locally_best_forest.predict(X_holdout_scaled))
))

MSE (cv): 494264.298
MSE (test): 440392.316


Feature Importance (Random Forest)

In [38]:
rf_importance = pd.DataFrame(
    locally_best_forest.best_estimator_.feature_importances_,
    columns=["importance"],
    index=X.columns
)

rf_importance.sort_values("importance", ascending=False)

,importance
temp,0.268264
yr,0.262016
atemp,0.225639
season,0.078362
hum,0.059876
mnth,0.031636
windspeed,0.030076
weathersit,0.025626
weekday,0.012105
workingday,0.004473
